In [4]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import time
import re

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "fr-FR,fr;q=0.9",
    "Referer": "https://www.transfermarkt.fr/",
}

def parse_valeur(val_str):
    if not val_str:
        return None
    val_str = val_str.replace("\xa0", "").replace(" ", "").replace("€", "")
    if "M" in val_str or "Mio" in val_str:
        return float(re.sub(r"[^\d,]", "", val_str).replace(",", ".")) * 1_000_000
    elif "k" in val_str or "Tsd" in val_str:
        return float(re.sub(r"[^\d,]", "", val_str).replace(",", ".")) * 1_000
    return None

def scrape_page(url, debug=False):
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")
    
    players = []
    rows = soup.select("table.items tbody tr.odd, table.items tbody tr.even")
    
    if debug and rows:
        # Affiche toutes les <td> de la première ligne pour identifier la structure
        print("=== DEBUG première ligne ===")
        for i, td in enumerate(rows[0].select("td")):
            print(f"  td[{i}] class={td.get('class')} | text='{td.text.strip()[:50]}'")
        print("===========================\n")
    
    for row in rows:
        try:
            tds = row.select("td")
            
            # Nom
            name_tag = row.select_one("td.hauptlink a")
            name = name_tag.text.strip() if name_tag else None
            
            # Poste (ligne sous le nom)
            pos_tag = row.select_one("td.hauptlink + td")
            position = pos_tag.text.strip() if pos_tag else None

            # Club
            club_tags = row.select("td.hauptlink a")
            club = club_tags[1].text.strip() if len(club_tags) > 1 else None
            
            # Nationalité
            nat_tag = row.select_one("img.flaggenrahmen")
            nationality = nat_tag["title"] if nat_tag else None

            # Âge — cherche la td qui contient uniquement un nombre à 2 chiffres
            age = None
            for td in tds:
                txt = td.text.strip()
                if re.fullmatch(r"\d{2}", txt):
                    age = int(txt)
                    break

            # Valeur marchande — dernière td.rechts
            val_tags = row.select("td.rechts")
            valeur_str = val_tags[-1].text.strip() if val_tags else None
            valeur = parse_valeur(valeur_str)

            players.append({
                "Joueur":      name,
                "Club":        club,
                "Nationalité": nationality,
                "Âge":         age,
                "Poste":       position,
                "Valeur_str":  valeur_str,
                "Valeur_EUR":  valeur,
            })

        except Exception as e:
            continue
    
    return players

# ── Test sur page 1 avec debug ───────────────────────────────────────────────
url_p1 = "https://www.transfermarkt.fr/ligue-1/marktwerte/wettbewerb/FR1/plus/0/galerie/0?page=1"
players = scrape_page(url_p1, debug=True)

df = pd.DataFrame(players)
print(df.head(10).to_string())

df.to_csv("transfermarkt_ligue1.csv", index=False)
print(f"\n✅ {len(df)} joueurs sauvegardés")
print(df.head(10))

=== DEBUG première ligne ===
  td[0] class=['zentriert'] | text='1'
  td[1] class=None | text='João Neves 


Milieu central'
  td[2] class=None | text=''
  td[3] class=['hauptlink'] | text='João Neves'
  td[4] class=None | text='Milieu central'
  td[5] class=['zentriert'] | text=''
  td[6] class=['zentriert'] | text='21'
  td[7] class=['zentriert'] | text=''
  td[8] class=['rechts', 'hauptlink'] | text='110,00 mio. €'

                  Joueur           Club Nationalité   Âge Poste     Valeur_str Valeur_EUR
0             João Neves  110,00 mio. €    Portugal  21.0  None  110,00 mio. €       None
1                Vitinha  110,00 mio. €    Portugal  26.0  None  110,00 mio. €       None
2        Ousmane Dembélé  100,00 mio. €      France  28.0  None  100,00 mio. €       None
3            Désiré Doué   90,00 mio. €      France  20.0  None   90,00 mio. €       None
4  Khvicha Kvaratskhelia   90,00 mio. €     Géorgie  25.0  None   90,00 mio. €       None
5          Achraf Hakimi   80,00 mio.